In [0]:
# Silver Layer: Inventory — Cleansing + SCD Type 1 MERGE

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, try_to_date, to_timestamp,
    when, coalesce, lit, current_timestamp,
    row_number, desc
)
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_inventory"
SILVER_TABLE = "ecomstore.ecomlake.silver_inventory"
TEMP_STAGING_TABLE = "ecomstore.ecomlake.temp_inventory_staging"

# Read from Bronze
bronze_df = spark.read.table(BRONZE_TABLE)

# Deduplication
w = Window.partitionBy("inventory_id").orderBy(desc("last_updated"), desc("_ingestion_timestamp"))

deduped_df = (
    bronze_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Cleansing & Standardization
cleaned_df = (
    deduped_df
    .withColumn("inventory_id", trim(upper(col("inventory_id"))))
    .withColumn("product_id",   trim(upper(col("product_id"))))
    .withColumn("warehouse_id", trim(upper(col("warehouse_id"))))
    .withColumn("stock_quantity",
        when(col("stock_quantity").cast(DoubleType()).cast(IntegerType()) >= 0,
             col("stock_quantity").cast(DoubleType()).cast(IntegerType()))
        .otherwise(None)   
    )
    .withColumn("reserved_quantity",
        when(col("reserved_quantity").cast(DoubleType()).cast(IntegerType()) >= 0,
             col("reserved_quantity").cast(DoubleType()).cast(IntegerType()))
        .otherwise(lit(0))
    )
    .withColumn("available_quantity",
        when(
            col("stock_quantity").isNotNull(),
            col("stock_quantity") - col("reserved_quantity")
        ).otherwise(None)
    )
    .withColumn("available_quantity",
        when(col("available_quantity") < 0, lit(0))
        .otherwise(col("available_quantity"))
    )
    .withColumn("reorder_level",
        when(col("reorder_level").cast(DoubleType()).cast(IntegerType()) > 0,
             col("reorder_level").cast(DoubleType()).cast(IntegerType()))
        .otherwise(lit(10))       
    )
    .withColumn("snapshot_date",
        coalesce(
            try_to_date(col("snapshot_date"), "yyyy-MM-dd"),
            try_to_date(col("snapshot_date"), "dd/MM/yyyy")
        )
    )
    .withColumn("last_updated",
        coalesce(
            to_timestamp(col("last_updated")),
            to_timestamp(col("last_updated"), "yyyy-MM-dd HH:mm:ss")
        )
    )
    .withColumn("is_below_reorder_level",
        when(
            col("stock_quantity").isNotNull() & col("reorder_level").isNotNull(),
            col("stock_quantity") <= col("reorder_level")
        ).otherwise(lit(False))
    )
    .withColumn("_silver_processed_at", current_timestamp())
    .filter(col("inventory_id").isNotNull() & col("product_id").isNotNull())
)

# Final Silver schema
silver_df = cleaned_df.select(
    "inventory_id", "product_id", "warehouse_id",
    "stock_quantity", "reserved_quantity", "available_quantity",
    "reorder_level",
    "snapshot_date", "last_updated",
    "is_below_reorder_level",
    "_batch_id", "_silver_processed_at"
)

# Disk Staging to prevent Double Compute
silver_df.write.format("delta").mode("overwrite").saveAsTable(TEMP_STAGING_TABLE)
staged_silver_df = spark.read.table(TEMP_STAGING_TABLE)

# Data Quality split
# Apply filters to the staged data
good_records = staged_silver_df.filter(
    col("inventory_id").isNotNull() &
    col("product_id").isNotNull() &
    col("warehouse_id").isNotNull() &
    col("stock_quantity").isNotNull()
)

bad_records = staged_silver_df.filter(
    col("inventory_id").isNull() |
    col("product_id").isNull() |
    col("warehouse_id").isNull() |
    col("stock_quantity").isNull()
)

(
    bad_records
    .withColumn("rejection_reason",
        when(col("inventory_id").isNull(),   lit("null_inventory_id"))
        .when(col("product_id").isNull(),    lit("null_product_id"))
        .when(col("warehouse_id").isNull(),  lit("null_warehouse_id"))
        .otherwise(lit("null_stock_quantity"))
    )
    .write.format("delta").mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("ecomstore.ecomlake.silver_quarantine_inventory")
)

# SCD Type 1 MERGE 
if spark.catalog.tableExists(SILVER_TABLE):
    silver_target = DeltaTable.forName(spark, SILVER_TABLE)

    (
        silver_target.alias("target")
        .merge(
            good_records.alias("source"),
            "target.inventory_id = source.inventory_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"✅ SCD Type 1 MERGE complete on {SILVER_TABLE}")

else:
    (
        good_records
        .write
        .format("delta")
        .mode("overwrite")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .partitionBy("warehouse_id") 
        .saveAsTable(SILVER_TABLE)
    )
    print(f"✅ Silver inventory table created: {SILVER_TABLE}")

# Clean up staging table
spark.sql(f"DROP TABLE IF EXISTS {TEMP_STAGING_TABLE}")
